# Clinical-LLM v2 — the longer training run, across several Colab sessions

Trains `configs/train_gpu_v2.yaml`: the v1 recipe for **2,100 steps** instead of 700.
That is about 9 hours on an L4, more than one Colab session allows, so the run is
split up. **Every session is the same: Runtime → Run all, then leave it.**

- Each session trains for `SESSION_HOURS`, saves a checkpoint to Google Drive and stops by itself.
- The next session picks up exactly where the last one stopped.
- The last cell tells you whether the run is finished or needs another session.

Before running: **Runtime → Change runtime type → GPU → L4 or A100** (a T4 is refused: it is too slow for this run).

> ⚠️ Research/education only — not medical advice, not a medical device. No patient data.

In [ ]:
#@title Settings  { display-mode: "form" }
SESSION_HOURS = 3.0   #@param {type:"number"}
# Colab reclaims runtimes at roughly 4.2-4.6 h. 3.0 leaves room for setup and
# for the checkpoint to reach Drive. Do not raise this above 3.5.

REPO_URL  = "https://github.com/asongwe-mutah/clinical-llm"
REPO_DIR  = "/content/clinical-llm"
CONFIG    = "configs/train_gpu_v2.yaml"
DATA_CFG  = "configs/data_gpu.yaml"
OUT       = "/content/drive/MyDrive/clinical-qlora-v2"   # NOT the v1 folder
EXPECTED_TRAIN_ROWS = 65_079
assert 0 < SESSION_HOURS <= 3.5, "keep SESSION_HOURS at 3.5 or below"

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU (L4 or A100)."
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability()
print(f"{name}  (compute capability {major}.{minor})")
if major < 8:
    raise SystemExit(
        f"\n{name} has no native bf16. This run would take days on it.\n"
        "Runtime > Change runtime type > pick L4 or A100, then Run all again."
    )

## 2. Google Drive (required)

Checkpoints live here. Without Drive a reclaimed runtime loses the whole session, so this cell stops if the mount fails. If it does, just run it again; the failure is usually temporary.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT, exist_ok=True)
probe = os.path.join(OUT, ".write_test")
with open(probe, "w") as f:      # mounted is not the same as writable
    f.write("ok")
os.remove(probe)

def _steps(prefix, sub=""):
    d = os.path.join(OUT, sub)
    if not os.path.isdir(d):
        return []
    return sorted(int(n.rsplit("-", 1)[-1]) for n in os.listdir(d) if n.startswith(prefix))

FINISHED = os.path.exists(os.path.join(OUT, "adapter_config.json"))
ck = _steps("checkpoint-")
print(f"output folder : {OUT}")
print(f"checkpoints   : {ck or 'none yet (this is the first session)'}")
print(f"milestones    : {_steps('step-', 'milestones') or 'none yet'}")
if FINISHED:
    print("\nThis run is ALREADY FINISHED. Nothing to train; skip to the last cell.")
elif ck:
    print(f"\nWill RESUME from step {ck[-1]}.")

## 3. Code and dependencies

In [ ]:
import subprocess, sys
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
assert os.path.exists(CONFIG), f"{CONFIG} missing: the clone is out of date."
print("code at", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

!pip install -q -r requirements.txt
!pip install -q -e .

# Colab preinstalls torchao 0.10; recent PEFT refuses any torchao older than 0.16
# when it builds LoRA layers. Nothing here uses torchao, so remove it.
_ta = subprocess.run([sys.executable, "-c", "import torchao; print(torchao.__version__)"],
                     capture_output=True, text=True)
if _ta.returncode == 0:
    from packaging.version import Version
    if Version(_ta.stdout.strip().split("+")[0]) < Version("0.16.0"):
        print(f"removing incompatible torchao {_ta.stdout.strip()}")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
import bitsandbytes as bnb; print("bitsandbytes", bnb.__version__)

## 4. Build the training corpus

Rebuilt every session (a new runtime starts empty). It must come out identical each time, or resuming would continue on different data, so the row count is checked.

In [ ]:
!python -m clinical_llm.data.prepare --config {DATA_CFG}
n_train = sum(1 for _ in open("data/processed/train.jsonl"))
print(f"train rows: {n_train:,}")
if n_train != EXPECTED_TRAIN_ROWS:
    raise SystemExit(
        f"\nExpected {EXPECTED_TRAIN_ROWS:,} training rows, got {n_train:,}.\n"
        "The corpus differs from the one this run was planned on. STOP: do not train.\n"
        "Send this message to Claude."
    )

## 5. Train for one session

A few minutes in, the log prints a **throughput probe** with the number of sessions the whole run needs. After that you can close the tab; the cell stops by itself at the session budget.

In [ ]:
if FINISHED:
    print("Run already finished; nothing to do.")
else:
    cmd = [sys.executable, "-m", "clinical_llm.train.train_qlora", "--config", CONFIG,
           "--output-dir", OUT, "--session-hours", str(SESSION_HOURS)]
    print(" ".join(cmd), "\n")
    !{" ".join(cmd)}

## 6. Where things stand

In [ ]:
import yaml
total = yaml.safe_load(open(CONFIG))["max_steps"]
ck = _steps("checkpoint-")
ms = _steps("step-", "milestones")
done = os.path.exists(os.path.join(OUT, "adapter_config.json"))
print(f"milestones saved : {ms or 'none yet'}")
print(f"newest checkpoint: {ck[-1] if ck else 'none'} of {total} steps")
if done:
    print("\n*** FINISHED. ***")
    print(f"In Google Drive, download the folder  {os.path.basename(OUT)}/milestones  to your Mac's")
    print("Downloads folder, then tell Claude it is there. Do not run any evaluation yourself.")
elif ck:
    left = total - ck[-1]
    print(f"\nNot finished: {left} steps to go.")
    print("Start a NEW session (Runtime > Disconnect and delete runtime, then Run all).")
    print("It resumes from the checkpoint above.")
else:
    print("\nNo checkpoint was written. Scroll up to the training cell and send the error to Claude.")